# Trực quan hóa review score — nối tiếp Phần 3 (SQL) và Phần 4 (Python)

Notebook này trực quan hóa các câu hỏi Q03–Q10 trong `03_analysis_SQL.ipynb` và `04_analysis_python.ipynb`. Nguồn dữ liệu dùng chung là SQLite view `v_review_order_features`, mỗi dòng tương ứng một đơn đã giao có review; cách này tránh đếm lặp theo sản phẩm hoặc thanh toán.

Notebook tạo hình về phân bố review score và mối liên hệ với giao hàng, phí vận chuyển, giỏ hàng, thanh toán, tháng đặt hàng, bang khách hàng. Tất cả ảnh được lưu vào `reports/figures/` để dùng trong báo cáo.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 130, "axes.titleweight": "bold"})


def find_project_root(start_dir: Path) -> Path:
    for candidate in (start_dir, *start_dir.parents):
        if (
            (candidate / "data" / "olist.db").is_file()
            and (candidate / "Notebooks").is_dir()
            and (candidate / "sql" / "03_queries.sql").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        f"Không tìm thấy thư mục gốc từ {start_dir}. "
        "Hãy chạy scripts/database/build_db.py trước."
    )


BASE_DIR = find_project_root(Path.cwd().resolve())
DB_PATH = BASE_DIR / "data" / "olist.db"
FIG_DIR = BASE_DIR / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(DB_PATH) as conn:
    available_views = {
        row[0]
        for row in conn.execute("SELECT name FROM sqlite_master WHERE type='view'")
    }
    if "v_review_order_features" not in available_views:
        raise RuntimeError(
            "Thiếu view v_review_order_features. "
            "Hãy chạy scripts/database/build_db.py để dựng lại database."
        )
    reviews = pd.read_sql_query(
        "SELECT * FROM v_review_order_features ORDER BY purchase_year_month, order_id",
        conn,
    )

reviews["is_late"] = reviews["is_late"].astype(bool)
reviews["low_score"] = reviews["review_score"] <= 2
reviews["positive_review"] = reviews["review_score"] >= 4
reviews["purchase_year_month"] = pd.to_datetime(
    reviews["purchase_year_month"], format="%Y-%m"
)

def save_figure(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, bbox_inches="tight", dpi=180)
    print(f"Đã lưu: {path.relative_to(BASE_DIR)}")
    plt.show()
    plt.close(fig)

print(f"Database: {DB_PATH}")
print(f"Đơn hàng có review: {len(reviews):,}")
print(f"Thư mục ảnh: {FIG_DIR}")


## Q03 — Phân bố và mất cân bằng review score

Cùng truy vấn Q03: số lượng và tỷ trọng đơn ở từng mức 1–5 sao. Tỷ trọng giúp nhìn rõ mức độ mất cân bằng của nhãn review.


In [ ]:
q03 = reviews.groupby("review_score").size().reindex(range(1, 6), fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(q03.index.astype(str), q03.values, color="#2878B5")
ax.bar_label(bars, labels=[f"{v / q03.sum():.1%}" for v in q03.values], padding=3)
ax.set(title="Q03 — Phân bố review score", xlabel="Điểm review (sao)", ylabel="Số đơn")
ax.margins(y=0.15)
fig.tight_layout()
save_figure(fig, "01_review_score_distribution.png")


## Q04 — Giao trễ và giao đúng hạn so với review

So sánh điểm trung bình, tỷ lệ review tích cực (≥4 sao) và review thấp (≤2 sao) giữa hai nhóm giao hàng, thống nhất với Q04 SQL.


In [ ]:
q04 = reviews.groupby("is_late").agg(
    order_count=("order_id", "size"),
    avg_review_score=("review_score", "mean"),
    positive_pct=("positive_review", "mean"),
    low_score_pct=("low_score", "mean"),
).rename(index={False: "Đúng hạn hoặc sớm", True: "Giao trễ"})
q04[["positive_pct", "low_score_pct"]] *= 100
print(q04.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
q04["avg_review_score"].plot(kind="bar", ax=axes[0], color=["#4C9F70", "#D9534F"])
axes[0].set(title="Điểm review trung bình", ylabel="Điểm (1–5)", xlabel="")
axes[0].set_ylim(0, 5)
axes[0].tick_params(axis="x", rotation=15)
q04[["positive_pct", "low_score_pct"]].plot(kind="bar", ax=axes[1], color=["#4C9F70", "#D9534F"])
axes[1].set(title="Tỷ lệ review theo nhóm", ylabel="Tỷ lệ (%)", xlabel="")
axes[1].tick_params(axis="x", rotation=15)
axes[1].legend(["Tích cực (≥4)", "Thấp (≤2)"])
fig.suptitle("Q04 — Tình trạng giao hàng và review score")
fig.tight_layout()
save_figure(fig, "02_late_vs_review_score.png")


## Q05 — Mức độ trễ theo ngày và review score

Dùng đúng các nhóm trễ trong Q05 SQL: sớm/đúng hạn, trễ 1–3 ngày, 4–7 ngày và trên 7 ngày.


In [ ]:
delay_labels = ["Sớm hoặc đúng hạn", "Trễ 1–3 ngày", "Trễ 4–7 ngày", "Trễ trên 7 ngày"]
reviews["delay_group"] = pd.cut(
    reviews["delay_days"], [-np.inf, 0, 3, 7, np.inf], labels=delay_labels
)
q05 = reviews.groupby("delay_group", observed=False).agg(
    order_count=("order_id", "size"),
    avg_review_score=("review_score", "mean"),
    positive_pct=("positive_review", "mean"),
)
q05["positive_pct"] *= 100
print(q05.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(q05.index.astype(str), q05["avg_review_score"], marker="o", linewidth=2.5, color="#D9534F")
ax.set(title="Q05 — Review score theo số ngày trễ", xlabel="Mức trễ", ylabel="Điểm review trung bình")
ax.set_ylim(0, 5)
ax.grid(axis="y", alpha=0.3)
ax2 = ax.twinx()
ax2.bar(q05.index.astype(str), q05["order_count"], alpha=0.15, color="#2878B5")
ax2.set_ylabel("Số đơn")
fig.tight_layout()
save_figure(fig, "03_delay_vs_review_score.png")


## Q06 — Phí ship và tỷ lệ phí/giá hàng

Chia đơn thành bốn quartile theo tổng phí ship và theo tỷ lệ phí ship/giá trị hàng như Q06 SQL. So sánh điểm review trung bình, đồng thời hiển thị cỡ mẫu.


In [ ]:
quartiles = ["Q1 thấp", "Q2", "Q3", "Q4 cao"]
reviews["freight_quartile"] = pd.qcut(reviews["total_freight"].rank(method="first"), 4, labels=quartiles)
valid_ratio = reviews.dropna(subset=["freight_to_price_pct"]).copy()
valid_ratio["ratio_quartile"] = pd.qcut(
    valid_ratio["freight_to_price_pct"].rank(method="first"), 4, labels=quartiles
)
q06_freight = reviews.groupby("freight_quartile", observed=False).agg(
    order_count=("order_id", "size"), avg_review_score=("review_score", "mean")
)
q06_ratio = valid_ratio.groupby("ratio_quartile", observed=False).agg(
    order_count=("order_id", "size"), avg_review_score=("review_score", "mean")
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, summary, title in [
    (axes[0], q06_freight, "Theo tổng phí ship"),
    (axes[1], q06_ratio, "Theo tỷ lệ phí/giá"),
]:
    bars = ax.bar(summary.index.astype(str), summary["avg_review_score"], color="#8172B2")
    ax.bar_label(bars, labels=[f"n={n:,}" for n in summary["order_count"]], padding=3, fontsize=8)
    ax.set(title=title, xlabel="Quartile", ylim=(0, 5))
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Điểm review trung bình (1–5)")
fig.suptitle("Q06 — Phí vận chuyển và review score")
fig.tight_layout()
save_figure(fig, "04_freight_vs_review_score.png")


## Q07 — Độ phức tạp đơn hàng

So sánh điểm review và tỷ lệ review thấp theo số mặt hàng trong giỏ và số người bán trong đơn, theo nhóm trong Q07 SQL.


In [ ]:
reviews["basket_group"] = np.select(
    [reviews["item_count"].eq(1), reviews["item_count"].between(2, 3)],
    ["1 mặt hàng", "2–3 mặt hàng"], default="4+ mặt hàng"
)
reviews["seller_group"] = np.where(reviews["seller_count"].eq(1), "1 người bán", "Nhiều người bán")
q07 = reviews.groupby(["basket_group", "seller_group"]).agg(
    order_count=("order_id", "size"),
    avg_review_score=("review_score", "mean"),
    low_score_pct=("low_score", "mean"),
)
q07["low_score_pct"] *= 100
print(q07.round(2))

score = q07["avg_review_score"].unstack("seller_group")
fig, ax = plt.subplots(figsize=(9, 4.5))
score.plot(kind="bar", ax=ax, color=["#2878B5", "#E07B39"])
ax.set(title="Q07 — Review theo quy mô giỏ và số người bán", xlabel="Số mặt hàng", ylabel="Điểm review trung bình")
ax.set_ylim(0, 5)
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Người bán")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, "05_order_complexity_vs_review_score.png")


## Q08 — Phương thức thanh toán và trả góp

Đối chiếu điểm trung bình, tỷ lệ review thấp và cỡ mẫu theo `first_payment_type`, như Q08 SQL.


In [ ]:
q08 = reviews.dropna(subset=["first_payment_type"]).groupby("first_payment_type").agg(
    order_count=("order_id", "size"),
    avg_installments=("max_installments", "mean"),
    avg_review_score=("review_score", "mean"),
    low_score_pct=("low_score", "mean"),
).sort_values("order_count", ascending=False)
q08["low_score_pct"] *= 100
print(q08.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_data = q08.sort_values("avg_review_score")
bars = ax.barh(plot_data.index, plot_data["avg_review_score"], color="#2878B5")
ax.bar_label(bars, labels=[f"{score:.2f} (n={n:,})" for score, n in zip(plot_data["avg_review_score"], plot_data["order_count"])], padding=4)
ax.set(title="Q08 — Review score theo phương thức thanh toán", xlabel="Điểm review trung bình", xlim=(0, 5))
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
save_figure(fig, "06_payment_vs_review_score.png")


## Q09 — Xu hướng review theo tháng

Thể hiện điểm review trung bình, tỷ lệ review thấp và số đơn theo tháng đặt hàng, tương ứng với Q09 SQL. Tháng đặt hàng là mốc nhất quán trong view; không diễn giải nó thành tháng review được gửi.


In [ ]:
q09 = reviews.groupby("purchase_year_month").agg(
    order_count=("order_id", "size"),
    avg_review_score=("review_score", "mean"),
    low_score_pct=("low_score", "mean"),
).sort_index()
q09["low_score_pct"] *= 100

fig, (ax_score, ax_count) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]}
)
ax_score.plot(q09.index, q09["avg_review_score"], marker="o", color="#2878B5", label="Điểm review trung bình")
ax_score.set(title="Q09 — Review score theo tháng đặt hàng", ylabel="Điểm review trung bình", ylim=(0, 5))
ax_score.grid(axis="y", alpha=0.25)
ax_rate = ax_score.twinx()
ax_rate.plot(q09.index, q09["low_score_pct"], marker="s", color="#D9534F", label="Review thấp (≤2 sao)")
ax_rate.set_ylabel("Review thấp (≤2 sao, %)")
ax_rate.set_ylim(0, 100)
lines = ax_score.get_lines() + ax_rate.get_lines()
ax_score.legend(lines, [line.get_label() for line in lines], loc="best")
ax_count.bar(q09.index, q09["order_count"], width=20, color="#8172B2", alpha=0.8)
ax_count.set_ylabel("Số đơn có review")
ax_count.grid(axis="y", alpha=0.25)
ax_count.tick_params(axis="x", rotation=45)
fig.tight_layout()
save_figure(fig, "07_monthly_review_score.png")

## Q10 — Mức hài lòng theo bang khách hàng

Quan sát điểm review, tỷ lệ giao trễ và tỷ lệ review thấp theo bang (chỉ giữ bang có ít nhất 100 đơn, như Q10 SQL). Kích thước điểm biểu thị số đơn; biểu đồ cho thấy liên hệ địa lý, không phải quan hệ nhân quả.


In [ ]:
q10 = reviews.groupby("customer_state").agg(
    order_count=("order_id", "size"),
    avg_review_score=("review_score", "mean"),
    low_score_pct=("low_score", "mean"),
    late_rate_pct=("is_late", "mean"),
)
q10 = q10.loc[q10["order_count"] >= 100].sort_values("avg_review_score")
q10[["low_score_pct", "late_rate_pct"]] *= 100
print(q10.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 9), sharey=True)
colors = plt.cm.coolwarm(plt.Normalize(q10["low_score_pct"].min(), q10["low_score_pct"].max())(q10["low_score_pct"]))
axes[0].barh(q10.index, q10["avg_review_score"], color=colors)
axes[0].set(title="Điểm review (màu = tỷ lệ thấp)", xlabel="Điểm review trung bình", xlim=(0, 5))
axes[0].set_ylabel("Bang khách hàng")
axes[0].grid(axis="x", alpha=0.25)
axes[1].barh(q10.index, q10["late_rate_pct"], color="#8172B2")
axes[1].set(title="Tình trạng giao hàng", xlabel="Tỷ lệ đơn giao trễ (%)")
axes[1].grid(axis="x", alpha=0.25)
colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(q10["low_score_pct"].min(), q10["low_score_pct"].max()), cmap="coolwarm"), ax=axes[0], pad=0.02)
colorbar.set_label("Review thấp (≤2 sao, %)")
fig.suptitle("Q10 — Mức hài lòng và giao trễ theo bang", y=1.01)
fig.tight_layout()
save_figure(fig, "08_review_by_state.png")

## Tổng kết và đầu ra

Tám biểu đồ trên theo cùng thứ tự Q03–Q10 với Phần 3 (SQL) và Phần 4 (Python). Ảnh được tạo lại tại `reports/figures/` với tên `01_review_score_distribution.png` đến `08_review_by_state.png`. Các khác biệt quan sát được là mối liên hệ mô tả; không nên kết luận quan hệ nhân quả.
